# Numerical Precision

In this section, you will investigate how different convolution
and matrix-matrix multiplication kernel performs when changing the
numerical precision.

## 1. Set-up

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [ ]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a4-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}"

In [ ]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

Verify the following cell prints your github repository.

In [ ]:
!ls {PROJECT_ROOT}

In [ ]:
!pip install torch numpy matplotlib

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.matmul import matmul
from src.conv2d import conv2d

# 2. Convolution

In the following cell(s), please plot the reconstruction error of an
approximated tensor (in the y-axis) with the numerical precision
(in the x-axis). Please show one plot for `winograd` and one plot for `fft`.

In [ ]:
# Set up test data
def generate_test_data(precision):
    dtype_map = {
        16: torch.float16,
        32: torch.float32, 
        64: torch.float64
    }
    dtype = dtype_map[precision]
    
    # Create input and kernel with random values
    x = torch.randn(
        1,
        3,
        32,
        32,
        dtype=dtype
    )
    kernel = torch.randn(
        16,
        3,
        3,
        3,
        dtype=dtype
    )
    
    return x, kernel

In [ ]:
# Calculate L1 reconstruction error
def calculate_error(naive_result, method_result):
    # L1 norm of the difference
    return torch.sum(
        torch.abs(naive_result - method_result)
    ).item()

In [ ]:
from src.conv2d import winograd
# TODO: plot the error v.s. precision curve
# Precision levels to test for Winograd

precisions_winograd = [16, 32, 64]
winograd_errors = []

# Run tests for Winograd at each precision
print("Running Winograd tests...")
for precision in precisions_winograd:
    x, kernel = generate_test_data(precision)
    
    # Naive (ground truth) implementation
    naive_result = conv2d(x, kernel, method='naive')
    
    # Winograd implementation
    winograd_result = conv2d(x, kernel, method='winograd')
    winograd_error = calculate_error(naive_result, winograd_result)
    winograd_errors.append(winograd_error)
    print(f"  float{precision} error: {winograd_error}")

# Plot Winograd results
plt.figure(figsize=(8, 5))
plt.bar([str(p) for p in precisions_winograd], winograd_errors)
plt.title('Winograd Reconstruction Error')
plt.xlabel('Numerical Precision (bits)')
plt.ylabel('L1 Reconstruction Error')
plt.tight_layout()
plt.show()

In [ ]:
from src.conv2d import fft
# TODO: plot the error v.s. precision curve

# Precision levels to test for FFT (no float16 support)
precisions_fft = [32, 64]
fft_errors = []

# Run tests for FFT at each precision
print("\nRunning FFT tests...")
for precision in precisions_fft:
    x, kernel = generate_test_data(precision)
    
    # Naive (ground truth) implementation
    naive_result = conv2d(x, kernel, method='naive')
    
    # FFT implementation
    fft_result = conv2d(x, kernel, method='fft')
    fft_error = calculate_error(naive_result, fft_result)
    fft_errors.append(fft_error)
    print(f"  float{precision} error: {fft_error}")

# Plot FFT results
plt.figure(figsize=(8, 5))
plt.bar([str(p) for p in precisions_fft], fft_errors)
plt.title('FFT Reconstruction Error')
plt.xlabel('Numerical Precision (bits)')
plt.ylabel('L1 Reconstruction Error')
plt.tight_layout()
plt.show()

# 3. Matrix-matrix Multiply

In the following cell(s), please plot the reconstruction error (in the y-axis)
with the different numerical precisions (in the x-axis) for `log` (i.e.
logorithmic matrix-matrix multiplication).

In [ ]:
# Set up test data
def generate_test_data(precision):
    dtype_map = {
        16: torch.float16,
        32: torch.float32, 
        64: torch.float64
    }
    dtype = dtype_map[precision]
    
    # Create random matrices for multiplication
    # Adjust dimensions as needed for your implementation
    A = torch.randn(64, 32, dtype=dtype)
    B = torch.randn(32, 48, dtype=dtype)
    
    return A, B

In [ ]:
from src.matmul import logmatmul

# TODO: plot the error v.s. precision curve

# Precision levels to test
precisions = [16, 32, 64]
log_errors = []

# Run tests for each precision
print("Running Log Matrix Multiplication tests...")
for precision in precisions:
    A, B = generate_test_data(precision)
    
    # Naive (ground truth) implementation
    naive_result = matmul(A, B, method='naive')
    
    # Log matrix multiplication implementation
    try:
        log_result = matmul(A, B, method='log')
        log_error = calculate_error(naive_result, log_result)
        log_errors.append(log_error)
        print(f"  float{precision} error: {log_error}")
    except Exception as e:
        print(f"  float{precision} failed: {e}")
        log_errors.append(0)  # or None, to indicate failure

# Plot log matrix multiplication results
plt.figure(figsize=(10, 6))
plt.bar([str(p) for p in precisions], log_errors)
plt.title('Log Matrix Multiplication Reconstruction Error')
plt.xlabel('Numerical Precision (bits)')
plt.ylabel('L1 Reconstruction Error')
plt.yscale('log')  # Using log scale for y-axis if errors vary greatly
plt.tight_layout()
plt.show()

# Print the numeric values for reference
print("\nLog Matrix Multiplication Errors:")
for p, e in zip(precisions, log_errors):
    print(f"  float{p}: {e}")